In [1]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

#Load block data from Jinping's code & fix time zones.
block_df = pd.read_parquet(os.path.join(helper.path_ptot,"PT_prior_post_imv.parquet"))
block_df['discharge_dttm'] = helper.ensure_datetime_est(block_df['discharge_dttm'])
block_df['block_vent_start_dttm'] = helper.ensure_datetime_est(block_df['block_vent_start_dttm'])
block_df['block_vent_end_dttm'] = helper.ensure_datetime_est(block_df['block_vent_end_dttm'])
block_df['block_first_vital_dttm'] = helper.ensure_datetime_est(block_df['block_first_vital_dttm'])
block_df['block_last_vital_dttm'] = helper.ensure_datetime_est(block_df['block_last_vital_dttm'])
block_df['death_dttm'] = helper.ensure_datetime_est(block_df['death_dttm'])
block_df['final_outcome_dttm'] = helper.ensure_datetime_est(block_df['final_outcome_dttm'])
block_df['encounter_block'] = block_df['encounter_block'].astype(str)

####TESTING ONLY####################################
#block_df = block_df.head(100) #smallercohort for development
#print("WARNING: Smaller cohort size for testing.")

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")
print(f"Unique Admission IDs: {block_df['hospitalization_id'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126
Unique Admission IDs: 32136


In [2]:
#ICU ADT data
#Load the CLIF tables
adt_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_adt.parquet"))

#Remove the ones not in cohort 
adt_df = adt_df[adt_df['hospitalization_id'].isin( block_df['hospitalization_id'] )].reset_index()

#fix timezone issues
adt_df['out_dttm'] = helper.ensure_datetime_est(adt_df['out_dttm'])
adt_df['in_dttm'] = helper.ensure_datetime_est(adt_df['in_dttm'])

#Merge with encounter block
merged_adt_df = pd.merge (
    block_df[['encounter_block','hospitalization_id','block_vent_start_dttm']],
    adt_df,
    on='hospitalization_id',
    how = 'inner'
)
#keep ICUs only
merged_adt_df = merged_adt_df[merged_adt_df['location_category']== 'icu']

#Sort by in_dttm
merged_adt_df= merged_adt_df.sort_values(['encounter_block','in_dttm'], ascending=True)

#ICU Type and In Time
#remove anywhere the patient left before start of IMV
ICU_df = merged_adt_df[merged_adt_df['out_dttm'] > merged_adt_df['block_vent_start_dttm']]
ICU_df = ICU_df.drop_duplicates(subset=['encounter_block'], keep='first')
ICU_df.rename(columns={'location_name':'ICU_type','in_dttm':'icu_in_dttm'}, inplace=True)
ICU_df = ICU_df[['encounter_block','ICU_type','icu_in_dttm']]
#Merge back to block
block_df = block_df.merge(
    ICU_df,
    on='encounter_block',
    how='left'
)
block_df['ICU_type'] = block_df['ICU_type'].astype(str)


#ICU Re-admission, post IMV
ICU_df = merged_adt_df[merged_adt_df['out_dttm'] > merged_adt_df['block_vent_start_dttm']]
ICU_df['icu_readmission'] = ICU_df['encounter_block'].duplicated(keep='first') #If an encounter block is duplicated after removing pre-IMV ICU stays, this implies re-admission.
icu_readmit_df = (
   ICU_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_readmission=("icu_readmission", "max"))
)
#Merge back to blocks
block_df = block_df.merge(
    icu_readmit_df[["encounter_block", "icu_readmission"]],
    on=["encounter_block"],
    how="left"
)
block_df["icu_readmission"] = block_df["icu_readmission"].astype(bool)

#ICU stays in total
#Whole encounter block, not just post-IMV
icu_N_df = (
    merged_adt_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_N=("in_dttm", "count"))
)
#Marge back to blocks
block_df = block_df.merge(
    icu_N_df[["encounter_block", "icu_N"]],
    on=["encounter_block"],
    how="left"
)

#ICU LOS, including whole encounter block, not just post IMV
merged_adt_df["icu_los_days"] = (merged_adt_df["out_dttm"] - merged_adt_df["in_dttm"]).dt.total_seconds() / (24*3600)
icu_los_df = (
    merged_adt_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_los_days=("icu_los_days", "sum"))
)
#Merge back to blocks
block_df = block_df.merge(
    icu_los_df[["encounter_block", "icu_los_days"]],
    on=["encounter_block"],
    how="left"
)

del adt_df, merged_adt_df, ICU_df
print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")


Block Length: 32276
Unique Encounter Block: 32126


/tmp/ipykernel_416478/1308271330.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ICU_df['icu_readmission'] = ICU_df['encounter_block'].duplicated(keep='first') #If an encounter block is duplicated after removing pre-IMV ICU stays, this implies re-admission.


In [4]:
"""
All of the CLIF tables are associated with a hospitalization_id without any patient_id data. We have these two data frames below.

block_df (From the CLIF Eligibility for Mobilization Output
encounter_block, hospitalization_id, patient_id
97, 123, X
97, 124, X
98, 125, X

hosp_df (CLIF)
hospitalization_id, patient_id
122, X
123, X
124, X
125, X
126, X

MERGE hosp_df and block_df on=patient how=left.

NEW hosp_df
hospitalization_id, patient_id, encounter_block
122, X, 97
123, X, 97
124, X, 97
125, X, 97
126, X, 97
122, X, 98
123, X, 98
124, X, 98
125, X, 98
126, X, 98
Note the duplicates are on purpose.

Now we can look at any CLIF table.
For a given encounter_block, it will pull data from any hospitalization_id associated with the same patient in the encounter_block.
Then we can filter based on date_time only wihtout any sort of filter based on hospitalization_id.
"""
#Load and filter Hospitalizations data from CLIF
hosp_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_hospitalization.parquet"))
hosp_df = hosp_df[hosp_df['patient_id'].isin(block_df['patient_id'])].reset_index()
hosp_df['admission_dttm'] = helper.ensure_datetime_est(hosp_df['admission_dttm'])
hosp_df['discharge_dttm'] =  helper.ensure_datetime_est(hosp_df['discharge_dttm'])
print(f"Hosp DF size: {len(hosp_df)}")

#Aadd encounter_block to the hospitalization data frame.
hosp_df = hosp_df.sort_values('admission_dttm')
hosp_df = hosp_df.merge(
    block_df[['patient_id','encounter_block','block_vent_start_dttm']].drop_duplicates(),
    on='patient_id',
    how='left').reset_index()

print(f"Hosp DF size: {len(hosp_df)}")

Hosp DF size: 104678
Hosp DF size: 137490


In [5]:
## FOR Sex, Race and Ethnicity, and Language

#Load and Filter Patient DF from CLIF
patient_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_patient.parquet"))
patient_df = patient_df[patient_df['patient_id'].isin ( block_df['patient_id'] )].reset_index()
patient_df = patient_df[['patient_id','sex_category','race_category','ethnicity_category','birth_date','language_category']]

#merge
block_df = pd.merge(
    block_df,
    patient_df,
    on='patient_id',
    how='left'
)

#Convert birth date to date object
if not helper.use_mimic:
    block_df['birth_date'] = pd.to_date(block_df['birth_date'], errors='coerce')
    block_df['age'] = block_df['block_vent_start_dttm'].dt.year - block_df['birth_date'].dt.year

#remove useless column
block_df = block_df.drop(columns=['birth_date'])

del patient_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [6]:
##AGE, LANGUAGE, Payor from MIMIC data given missing on CLIF'd tables.
"""
Language and Payor for MIMIC is extracted from the admissions table.
Our CLIF'd tables are missing birth date since this is not in MIMIC but rather uses anchor_year and anchor_age.
We are calculating age at the time of intubation.
Also, note that MIMIC age stops at 89, all patients older than 89 are assiged anchor_age 91.
Language for MIMIC is extracted from the admissions table.
"""
if helper.use_mimic:
    #AGE
    
    #load
    patient_mimic_df = pd.read_parquet(os.path.join(helper.path_mimic,"hosp","patients.parquet"))
    patient_mimic_df.rename(columns={'subject_id': 'patient_id'}, inplace=True)
    patient_mimic_df['patient_id'] = patient_mimic_df['patient_id'].astype(str)
    
    #filter
    patient_mimic_df = patient_mimic_df[patient_mimic_df['patient_id'].isin( block_df['patient_id'] )]
    print(f"Unique MIMIC patient id: {patient_mimic_df['patient_id'].nunique()}")
    print(f"Unique Block patient id: {block_df['patient_id'].nunique()}")
    
    #merge
    merged_patient_df = pd.merge(
        block_df,
        patient_mimic_df,
        on='patient_id',
        how='left'
    )
    #convert admission time to year and calculate year - anchor_year + anchor_age
    merged_patient_df['age'] = merged_patient_df['block_vent_start_dttm'].dt.year - merged_patient_df['anchor_year'] + merged_patient_df['anchor_age']

    #Date of death
    merged_patient_df['date_of_death'] = pd.to_datetime(merged_patient_df['dod'])
    merged_patient_df['date_of_death'] = helper.ensure_datetime_est(merged_patient_df['date_of_death'])

    #Admission Year (based on estimate from anchor year
    merged_patient_df["anchor_year_group"] = merged_patient_df["anchor_year_group"].str.slice(0, 4).astype(int) + 1 #Convert achor year group to an integer
    merged_patient_df["admission_year"] = block_df['block_first_vital_dttm'].dt.year.astype(int) - merged_patient_df["anchor_year"] + merged_patient_df["anchor_year_group"]
    
    #drop unnecessary columns
    block_df = merged_patient_df.drop(columns=['gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'])
    
    ###LANGUAGE & PAYOR
    
    #For Language, we need to get this from a different table.
    block_df = block_df.drop(columns=['language_category'])
    
    #load
    adm_mimic_df = pd.read_parquet(os.path.join(helper.path_mimic,"hosp","admissions.parquet"))
    adm_mimic_df.rename(columns={'hadm_id': 'hospitalization_id'}, inplace=True)
    adm_mimic_df['hospitalization_id'] = adm_mimic_df['hospitalization_id'].astype(str)
    adm_mimic_df['mimic_race'] = adm_mimic_df['race'].astype(str)
    adm_mimic_df['insurance'] = adm_mimic_df['insurance'].astype(str)
    adm_mimic_df['language'] = adm_mimic_df['language'].astype(str)
    adm_mimic_df = adm_mimic_df[['hospitalization_id','language','insurance','mimic_race']]
    
    #filter
    adm_mimic_df = adm_mimic_df[adm_mimic_df['hospitalization_id'].isin( block_df['hospitalization_id'] )]
    print(f"Unique MIMIC hosp id: {adm_mimic_df['hospitalization_id'].nunique()}")
    print(f"Unique Block patient id: {block_df['hospitalization_id'].nunique()}")
    
    #merge
    block_df = pd.merge(
        block_df,
        adm_mimic_df,
        on='hospitalization_id',
        how='left'
    )

    print("MIMIC Race:")
    print(block_df['mimic_race'].value_counts())

    del patient_mimic_df, merged_patient_df, adm_mimic_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Unique MIMIC patient id: 29396
Unique Block patient id: 29396
Unique MIMIC hosp id: 32136
Unique Block patient id: 32136
MIMIC Race:
mimic_race
WHITE                                        19173
UNKNOWN                                       4480
BLACK/AFRICAN AMERICAN                        2314
OTHER                                         1092
UNABLE TO OBTAIN                               994
WHITE - OTHER EUROPEAN                         782
HISPANIC/LATINO - PUERTO RICAN                 402
ASIAN                                          346
ASIAN - CHINESE                                320
HISPANIC/LATINO - DOMINICAN                    263
WHITE - RUSSIAN                                247
PATIENT DECLINED TO ANSWER                     232
HISPANIC OR LATINO                             224
BLACK/CAPE VERDEAN                             184
BLACK/CARIBBEAN ISLAND                         184
PORTUGUESE                                     139
BLACK/AFRICAN                           

In [7]:
# Add admit_dttm and admit_type_name from hospitalization table

#Merge back to block_df
block_df = pd.merge(
    block_df,
    hosp_df[["patient_id", "hospitalization_id", "admission_dttm", "admission_type_name"]].drop_duplicates(),
    on=["patient_id", "hospitalization_id"],
    how='left'
)

block_df.rename(columns={"admission_type_name":"admission_source"}, inplace=True)

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}") 

Block Length: 32276
Unique Encounter Block: 32126


In [8]:
#Load the Vitals CLIF tables
vitals_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_vitals.parquet"))

#Remove the ones not in cohort, add encounter_block and vent start time as reference
merged_vitals_df = vitals_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#fix timezone issues
merged_vitals_df['recorded_dttm'] = helper.ensure_datetime_est(merged_vitals_df['recorded_dttm'])

#Calculate time windows from block_vent_start_dttm
merged_vitals_df['time_diff'] = merged_vitals_df['recorded_dttm'] - merged_vitals_df['block_vent_start_dttm']
merged_vitals_df['time_window'] = merged_vitals_df['time_diff'].apply(helper.classify_time_window)

In [9]:
#MIMIC ED Vitals Signs
if helper.use_mimic:
    print(f"Old length of vitals list: {len(merged_vitals_df)}")
    
    mimiced_vitals_df = pd.read_csv(os.path.join(helper.path_mimic,"ed","vitalsign.csv.gz"))
    mimiced_vitals_df.rename(columns={'subject_id':'patient_id','charttime':'recorded_dttm'},inplace=True)
    mimiced_vitals_df = mimiced_vitals_df.drop(['stay_id','temperature','rhythm','pain','o2sat'], axis=1)
    mimiced_vitals_df['patient_id'] = mimiced_vitals_df['patient_id'].astype(str)
    mimiced_vitals_df['recorded_dttm'] = helper.ensure_datetime_est(mimiced_vitals_df['recorded_dttm'])
    mimiced_vitals_df = mimiced_vitals_df.merge(block_df[['encounter_block','patient_id','block_vent_start_dttm']],
                                                       on='patient_id', how='right').reset_index()
    
    mimiced_vitals_df['time_diff'] = mimiced_vitals_df['recorded_dttm'] - mimiced_vitals_df['block_vent_start_dttm']
    mimiced_vitals_df['time_window'] = mimiced_vitals_df['time_diff'].apply(helper.classify_time_window)
    mimiced_vitals_df = mimiced_vitals_df[mimiced_vitals_df['time_window'].notna()].reset_index()

    #For heart rate
    mini_df = mimiced_vitals_df.drop(['resprate','sbp','dbp'], axis=1)
    mini_df.rename(columns={'heartrate':'vital_value'},inplace=True)
    mini_df['vital_category'] = 'heart_rate'
    mini_df = mini_df.dropna(subset=['vital_value'])
    merged_vitals_df = pd.concat([merged_vitals_df,mini_df],ignore_index=True, join='inner')
    del mini_df

    #For resp rate
    mini_df = mimiced_vitals_df.drop(['heartrate','sbp','dbp'], axis=1)
    mini_df.rename(columns={'resprate':'vital_value'},inplace=True)
    mini_df['vital_category'] = 'respiratory_rate'
    mini_df = mini_df.dropna(subset=['vital_value'])
    merged_vitals_df = pd.concat([merged_vitals_df,mini_df],ignore_index=True, join='inner')
    del mini_df

    #For sbp
    mini_df = mimiced_vitals_df.drop(['heartrate','resprate','dbp'], axis=1)
    mini_df.rename(columns={'sbp':'vital_value'},inplace=True)
    mini_df['vital_category'] = 'sbp'
    mini_df = mini_df.dropna(subset=['vital_value'])
    merged_vitals_df = pd.concat([merged_vitals_df,mini_df],ignore_index=True, join='inner')
    del mini_df

    #For map
    mimiced_vitals_df['vital_value'] = (0.333*mimiced_vitals_df['sbp']) + (0.666*mimiced_vitals_df['dbp'])
    mini_df = mimiced_vitals_df.drop(['heartrate','resprate','sbp','dbp'], axis=1)
    mini_df['vital_category'] = 'map'
    mini_df = mini_df.dropna(subset=['vital_value'])
    merged_vitals_df = pd.concat([merged_vitals_df,mini_df],ignore_index=True, join='inner')
    del mini_df, mimiced_vitals_df

    print(f"New length of vitals list: {len(merged_vitals_df)}")
    

Old length of vitals list: 51916697
New length of vitals list: 52077533


In [10]:
'''
#Exploration into different time windows to address missingness
def time_window_test(hours):
    if -24 <= hours < 0:
        return 'prior24h'
    elif 0 <= hours <= 1:
        return '0-1h'
    elif 1 < hours <= 4:
        return '1-3h'
    else:
        return None

limited_vitals = merged_vitals_df[['encounter_block','vital_category','time_diff']].copy()
limited_vitals['time_diff'] = limited_vitals['time_diff'].dt.total_seconds()/3600
limited_vitals['time_window'] = limited_vitals['time_diff'].apply(time_window_test)

pivot_df = pd.pivot_table (
    limited_vitals,
    values='encounter_block',
    index = ['time_window'],
    columns='vital_category',
    aggfunc='nunique'
)
del limited_vitals
print(pivot_df)
'''

"\n#Exploration into different time windows to address missingness\ndef time_window_test(hours):\n    if -24 <= hours < 0:\n        return 'prior24h'\n    elif 0 <= hours <= 1:\n        return '0-1h'\n    elif 1 < hours <= 4:\n        return '1-3h'\n    else:\n        return None\n\nlimited_vitals = merged_vitals_df[['encounter_block','vital_category','time_diff']].copy()\nlimited_vitals['time_diff'] = limited_vitals['time_diff'].dt.total_seconds()/3600\nlimited_vitals['time_window'] = limited_vitals['time_diff'].apply(time_window_test)\n\npivot_df = pd.pivot_table (\n    limited_vitals,\n    values='encounter_block',\n    index = ['time_window'],\n    columns='vital_category',\n    aggfunc='nunique'\n)\ndel limited_vitals\nprint(pivot_df)\n"

In [11]:
##HEART RATE

#Filter vitals_df for heart_rate only and merge
hr_df = merged_vitals_df[merged_vitals_df['vital_category'] == 'heart_rate'].copy()

#Group by encounter block and by time window, calculate mean
grouped_hr_df = (
    hr_df.groupby(['encounter_block', 'time_window'])['vital_value']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_hr_df.rename(columns={
    'prior24h': 'mean_heart_rate_prior24h',
    '0-24h': 'mean_heart_rate_0_24h',
    '24-48h': 'mean_heart_rate_24_48h',
    '48-72h': 'mean_heart_rate_48_72h',
}, inplace=True)

#Add back to encounter blocks
block_df = pd.merge(
    block_df,
    grouped_hr_df,
    on='encounter_block',
    how='left'
)

del hr_df, grouped_hr_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [12]:
##RESPIRATORY RATE
#Used to calculate qSOFA so will take worse (max) value.

#Filter vitals_df for heart_rate only and merge
rr_df = merged_vitals_df[merged_vitals_df['vital_category'] == 'respiratory_rate'].copy()

#Group by encounter block and by time window, calculate max
grouped_rr_df = (
    rr_df.groupby(['encounter_block', 'time_window'])['vital_value']
    .max()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_rr_df.rename(columns={
    'prior24h': 'max_resp_rate_prior24h',
    '0-24h': 'max_resp_rate_0_24h',
    '24-48h': 'max_resp_rate_24_48h',
    '48-72h': 'max_resp_rate_48_72h',
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_rr_df,
    on='encounter_block',
    how='left'
)

del rr_df, grouped_rr_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [13]:
##MAP

#First create MAP out of SBP and DBP when MAP is missing.

#Filter vitals_df to only relevant categories
bp_df = merged_vitals_df[
    merged_vitals_df['vital_category'].isin(['sbp', 'dbp', 'map'])
].copy()

#Pivot to get sbp, dbp, and map in columns per EB + recorded_dttm
pivoted_bp_df = bp_df.pivot_table(
    index=['encounter_block', 'recorded_dttm','time_window'],
    columns='vital_category',
    values='vital_value',
    aggfunc='first'  # in case there are multiple values per timestamp, take the first
).reset_index()

#Identify rows where MAP is missing but SBP and DBP are present
mask_missing_map = (
    pivoted_bp_df['map'].isna() &
    pivoted_bp_df['sbp'].notna() &
    pivoted_bp_df['dbp'].notna()
)

#Compute synthetic MAP
synthetic_map_rows = pivoted_bp_df[mask_missing_map].copy()
synthetic_map_rows['vital_value'] = (
    synthetic_map_rows['sbp'] + 2 * synthetic_map_rows['dbp']
) / 3
synthetic_map_rows['vital_category'] = 'map'

#Keep only necessary columns for synthetic rows
synthetic_map_rows = synthetic_map_rows[[
    'encounter_block', 'recorded_dttm', 'vital_category', 'vital_value','time_window'
]]

#Get existing 'map' rows (real ones)
real_map_rows = bp_df[
    bp_df['vital_category'] == 'map'
][[
    'encounter_block', 'recorded_dttm', 'vital_category', 'vital_value','time_window'
]]

#Combine real and synthetic MAP rows
map_df = pd.concat([real_map_rows, synthetic_map_rows], ignore_index=True)

#Group by encounter block and by time window, calculate mean
grouped_map_df = (
    map_df.groupby(['encounter_block', 'time_window'])['vital_value']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_map_df.rename(columns={
    'prior24h': 'mean_map_prior24h',
    '0-24h': 'mean_map_0_24h',
    '24-48h': 'mean_map_24_48h',
    '48-72h': 'mean_map_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_map_df,
    on='encounter_block',
    how='left'
)

del bp_df, pivoted_bp_df, synthetic_map_rows, real_map_rows, map_df, grouped_map_df,

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [14]:
##SYSTOLIC BP
#Used to calculate qSOFA so will take the minimum value

sbp_df = merged_vitals_df[merged_vitals_df['vital_category'] == 'sbp'].copy()

#Group by encounter block and by time window, calculate minimum
grouped_sbp_df = (
    sbp_df.groupby(['encounter_block', 'time_window'])['vital_value']
    .min()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_sbp_df.rename(columns={
    'prior24h': 'min_sbp_prior24h',
    '0-24h': 'min_sbp_0_24h',
    '24-48h': 'min_sbp_24_48h',
    '48-72h': 'min_sbp_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_sbp_df,
    on='encounter_block',
    how='left'
)

del sbp_df, grouped_sbp_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [15]:
#BMI

#Filter vitals table for height and weight only
bmi_df = vitals_df[
    vitals_df['vital_category'].isin(['height_cm', 'weight_kg'])
].copy()
#Add patient ID
bmi_df = pd.merge(bmi_df,hosp_df[['hospitalization_id','patient_id']],on='hospitalization_id',how='inner')

# Remove outliers
# For height rows: set out-of-range to NaN
is_height = bmi_df['vital_category'] == 'height_cm'
height_mask_low  = is_height & (bmi_df['vital_value'] < 76)
height_mask_high = is_height & (bmi_df['vital_value'] > 255)
bmi_df.loc[height_mask_low | height_mask_high, 'vital_value'] = np.nan

# For weight rows: set out-of-range to NaN
is_weight = bmi_df['vital_category'] == 'weight_kg'
weight_mask_low  = is_weight & (bmi_df['vital_value'] < 20)
weight_mask_high = is_weight & (bmi_df['vital_value'] > 1100)
bmi_df.loc[weight_mask_low | weight_mask_high, 'vital_value'] = np.nan

# Merge with vent_start_end to get ventilation start time
merged_bmi_df = pd.merge(
    block_df[['encounter_block','patient_id','block_vent_start_dttm']],
    bmi_df,
    on='patient_id',
    how='left'
)

# Calculate time difference between recorded_dttm and vent_start_time
merged_bmi_df['time_diff'] = (merged_bmi_df['recorded_dttm'] - merged_bmi_df['block_vent_start_dttm']).dt.total_seconds()

# Define whether measurement is before or after vent_start_time
merged_bmi_df['before_vent_start'] = (merged_bmi_df['time_diff'] <= 0).astype(int)

# Calculate absolute time difference
merged_bmi_df['abs_time_diff'] = merged_bmi_df['time_diff'].abs()

# Sort data to prioritize measurements before vent start and closest in time
merged_bmi_df = merged_bmi_df.sort_values(['encounter_block', 'vital_category', 'before_vent_start', 'abs_time_diff'], 
                                    ascending=[True, True, False, True])

# Drop duplicates to keep the closest measurement for each vital_category per encounter block
merged_bmi_df = merged_bmi_df.drop_duplicates(subset=['encounter_block', 'vital_category'], keep='first')

# Pivot to get height and weight per encounter block
pivot_bmi_df = merged_bmi_df.pivot(index='encounter_block', 
                                    columns='vital_category', 
                                    values='vital_value'
                                    ).reset_index()

# Calculate BMI
pivot_bmi_df['bmi'] = pivot_bmi_df['weight_kg'] / ((pivot_bmi_df['height_cm'] / 100) ** 2)

#Add back to block
block_df = pd.merge(
    block_df,
    pivot_bmi_df,
    on='encounter_block',
    how='left'
)

del bmi_df, merged_bmi_df, pivot_bmi_df, vitals_df, merged_vitals_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [16]:
#Load respiratory CLIF tables
resp_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_respiratory_support.parquet"))

#Remove the ones not in cohort and other not useful columns, add encounter_block and vent start time as reference
merged_resp_df = resp_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#fix timezone issues
merged_resp_df['recorded_dttm'] = helper.ensure_datetime_est(merged_resp_df['recorded_dttm'])

#Calculate time windows
merged_resp_df['time_diff'] = merged_resp_df['recorded_dttm'] - merged_resp_df['block_vent_start_dttm']
merged_resp_df['time_window'] = merged_resp_df['time_diff'].apply(helper.classify_time_window)
merged_resp_df = merged_resp_df[merged_resp_df['time_window'].notnull()]


In [17]:
#FiO2
fio2_df = merged_resp_df.copy()

#Set all values to be from 0 to 1
def normalize_fio2(x):
    if 20 < x < 100:
        return x/100
    elif 0.2 < x < 1:
        return x
    else:
        return np.nan
fio2_df['fio2_set'] = fio2_df['fio2_set'].apply(normalize_fio2)

#Fill in missing FiO2 for Other Modes
def synthetic_fio2(given, lpm, mode):
    #Delivery mode is more important than set FiO2
    #Room air is 21%
    if mode == "Room Air":
        return 0.21
    #NC is 20% + 4%xFlow if flow is given and within range. Otherwise it is NA.
    elif mode == "Nasal Cannula":
        if 0.5 < lpm <= 6:
            return 0.2 + 0.04*lpm
        else:
            return np.nan
    #Face mask, will have to assume simple since they do not mention NRB. Set to 5% + 5% x Flow but only if flow is within reasonable range. Else return
    elif mode == "Face Mask":
        if given:
            return given
        if 5 <= lpm <= 10:
            return 0.05 + 0.05*lpm
        else:
            return np.nan
    #Trach collar will assume room air if no specific FiO2 is given.
    elif mode == "Trach Collar":
        if given:
            return given
        else:
            return 0.21
    else: #This would include IMV and NIPPV.
        return given
fio2_df['fio2_set'] = fio2_df.apply(lambda row: synthetic_fio2(row['fio2_set'],row['lpm_set'],row['device_category']), axis=1)

#Remove missing data and keep only relevant columns
fio2_df = fio2_df[['encounter_block','time_window','fio2_set']].drop_duplicates()

#Group by encounter block and by time window, calculate mean for Fio2
grouped_fio2_df = (
    fio2_df.groupby(['encounter_block', 'time_window'])['fio2_set']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)
#Rename columns for clarity
grouped_fio2_df.rename(columns={
    'prior24h': 'mean_fio2_prior24h',
    '0-24h': 'mean_fio2_0_24h',
    '24-48h': 'mean_fio2_24_48h',
    '48-72h': 'mean_fio2_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_fio2_df,
    on='encounter_block',
    how='left'
)

del fio2_df, grouped_fio2_df

In [18]:
#PEEP set

#Group by encounter block and by time window, calculate mean for PEEP
grouped_peep_df = (
    merged_resp_df.groupby(['encounter_block', 'time_window'])['peep_set']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)
#Rename columns for clarity
grouped_peep_df.rename(columns={
    'prior24h':'mean_peep_prior24h',
    '0-24h': 'mean_peep_0_24h',
    '24-48h': 'mean_peep_24_48h',
    '48-72h': 'mean_peep_48_72h'
}, inplace=True)
block_df = pd.merge(
    block_df,
    grouped_peep_df,
    on='encounter_block',
    how='left'
)

del merged_resp_df, grouped_peep_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [19]:
#SOFA post intubation score
#The mobilization code already calculated for the first 24 hours. Other data could be extratected if desired. There is a dedicated python script to do this in their code.
sofa_df = pd.read_parquet(os.path.join(helper.path_mob,"intermediate","sofa.parquet"))
sofa_df = sofa_df[['encounter_block','sofa_total']]
sofa_df['encounter_block'] = sofa_df['encounter_block'].astype(str)
sofa_df.rename(columns={'sofa_total':'sofa_0_24h'},inplace=True)
block_df = pd.merge(
    block_df,
    sofa_df,
    on='encounter_block',
    how='left'
)
del sofa_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [20]:
#Load Assessment CLIF tables
ass_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_patient_assessments.parquet"))
ass_df = ass_df[ass_df['assessment_category'].isin(['braden_mobility','RASS','gcs_total','cam_total'])] #Make it smaller by using only the categories we want.

#Remove the ones not in cohort, add encounter_block and vent start time as reference
merged_ass_df = ass_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#fix timezone issues
merged_ass_df['recorded_dttm'] = helper.ensure_datetime_est(merged_ass_df['recorded_dttm'])

#Calculate time windows from block_vent_start_dttm
merged_ass_df['time_diff'] = merged_ass_df['recorded_dttm'] - merged_ass_df['block_vent_start_dttm']
merged_ass_df['time_window'] = merged_ass_df['time_diff'].apply(helper.classify_time_window)
merged_ass_df = merged_ass_df[merged_ass_df['time_window'].notnull()]

In [21]:
#Braden Mobility Score

#Filter ass_df for braden_mob only
braden_df = merged_ass_df[merged_ass_df['assessment_category'] == 'braden_mobility'].copy()

#Group by encounter block and by time window, calculate mean
grouped_braden_df = (
    braden_df.groupby(['encounter_block', 'time_window'])['numerical_value']
    .max()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_braden_df.rename(columns={
    'prior24h':'max_braden_prior24h',
    '0-24h': 'max_braden_0_24h',
    '24-48h': 'max_braden_24_48h',
    '48-72h': 'max_braden_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_braden_df,
    on='encounter_block',
    how='left'
)

del braden_df, grouped_braden_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [22]:
#RASS

#Filter ass_df for RASS only and merge
rass_df = merged_ass_df[merged_ass_df['assessment_category'] == 'RASS'].copy()

#Group by encounter block and by time window, calculate mean
grouped_rass_df = (
    rass_df.groupby(['encounter_block', 'time_window'])['numerical_value']
    .min()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_rass_df.rename(columns={
    'prior24h': 'min_RASS_prior24h',
    '0-24h': 'min_RASS_0_24h',
    '24-48h': 'min_RASS_24_48h',
    '48-72h': 'min_RASS_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_rass_df,
    on='encounter_block',
    how='left'
)

del rass_df, grouped_rass_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [23]:
#CAM-ICU

#Filter ass_df for braden_mob only
cam_df = merged_ass_df[merged_ass_df['assessment_category'] == 'cam_total'].copy()
cam_df['cam_pos'] = np.where(cam_df['categorical_value'] == 'Positive',1,0)

#Group by encounter block and by time window, calculate mean
grouped_cam_df = (
    cam_df.groupby(['encounter_block', 'time_window'])['cam_pos']
    .max()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_cam_df.rename(columns={
    'prior24h':'cam_icu_prior24h',
    '0-24h': 'cam_icu_0_24h',
    '24-48h': 'cam_icu_24_48h',
    '48-72h': 'cam_icu_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_cam_df,
    on='encounter_block',
    how='left'
)
block_df['cam_icu_prior24h'] = block_df['cam_icu_prior24h'].astype(bool)
block_df['cam_icu_0_24h'] = block_df['cam_icu_0_24h'].astype(bool)
block_df['cam_icu_24_48h'] = block_df['cam_icu_24_48h'].astype(bool)
block_df['cam_icu_48_72h'] = block_df['cam_icu_48_72h'].astype(bool)

del cam_df, grouped_cam_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [24]:
#GCS
#Used for qSOFA too, will take worse which is minimum.

#Filter ass_df for RASS only and merge
gcs_df = merged_ass_df[merged_ass_df['assessment_category'] == 'gcs_total'].copy()

#Group by encounter block and by time window, calculate mean
grouped_gcs_df = (
    gcs_df.groupby(['encounter_block', 'time_window'])['numerical_value']
    .min()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_gcs_df.rename(columns={
    'prior24h': 'min_GCS_prior24h',
    '0-24h': 'min_GCS_0_24h',
    '24-48h': 'min_GCS_24_48h',
    '48-72h': 'min_GCS_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_gcs_df,
    on='encounter_block',
    how='left'
)

del gcs_df, grouped_gcs_df, merged_ass_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [25]:
#qSOFA calculated using the values from above

block_df['qSOFA_prior24h'] = (block_df['min_GCS_prior24h'] < 15).astype(int) + \
    (block_df['max_resp_rate_prior24h'] >= 22).astype(int) + \
    (block_df['min_sbp_prior24h'] <= 100).astype(int)

block_df['qSOFA_0_24h'] = (block_df['min_GCS_0_24h'] < 15).astype(int) + \
    (block_df['max_resp_rate_0_24h'] >= 22).astype(int) + \
    (block_df['min_sbp_0_24h'] <= 100).astype(int)

block_df['qSOFA_24_48h'] = (block_df['min_GCS_24_48h'] < 15).astype(int) + \
    (block_df['max_resp_rate_24_48h'] >= 22).astype(int) + \
    (block_df['min_sbp_24_48h'] <= 100).astype(int)

block_df['qSOFA_48_72h'] = (block_df['min_GCS_48_72h'] < 15).astype(int) + \
    (block_df['max_resp_rate_48_72h'] >= 22).astype(int) + \
    (block_df['min_sbp_48_72h'] <= 100).astype(int)


In [26]:
#Elixhauser
#Using package comorbidipy with Quan et al mappings and Van Walraven weights
#Outputs both unadjusted and age adjusted.
import comorbidipy

if helper.use_mimic:
    diag_mimic_df = pd.read_csv(os.path.join(helper.path_mimic,"hosp","diagnoses_icd.csv.gz"))
    diag_mimic_df.rename(columns={'hadm_id':'hospitalization_id'}, inplace=True)
    diag_mimic_df['hospitalization_id'] = diag_mimic_df['hospitalization_id'].astype(str)
    diag_mimic_df['icd_version'] = diag_mimic_df['icd_version'].astype(int)

    #filter out primary diagnosis
    diag_mimic_df = diag_mimic_df[diag_mimic_df['seq_num'] != 1] #Remove primary diagnosis

    #merge with encounter blocks
    diag_df = pd.merge(
        block_df[['encounter_block','hospitalization_id','admission_year','age']],
        diag_mimic_df[['hospitalization_id','icd_code','icd_version']],
        on='hospitalization_id',
        how='left'
    )
    del diag_mimic_df
    diag_df = diag_df.drop('hospitalization_id',axis=1)
    diag_df.rename(columns={'icd_code':'code','encounter_block':'id'},inplace=True) #Need to change name for comorbidity function

    #Create separate data frames for each ICD version
    diag_9_df = diag_df[diag_df['icd_version'] == 9].drop_duplicates().reset_index().drop('icd_version',axis=1)
    diag_10_df = diag_df[diag_df['icd_version'] == 10].drop_duplicates().reset_index().drop('icd_version',axis=1)
    
    #Check how many encnounterblocks are in both datasets and report it as an issue.
    duplicate_icds_n = sum(diag_9_df['id'].isin(diag_10_df['id']))
    print(f"Number of encnounters with comorbidities in both ICD codes: {duplicate_icds_n}")

    #Run it for ICD 9
    elix_9_df = comorbidipy.comorbidity(
        diag_9_df,
        score = 'elixhauser',
        icd = 'icd9',
        variant = 'quan',
        weighting = 'vw',#Van Walraven weights
    )

    #Run it for ICD 10
    elix_10_df = comorbidipy.comorbidity(
        diag_10_df,
        score = 'elixhauser',
        icd = 'icd10',
        variant = 'quan',
        weighting = 'vw',#Van Walraven weights
    )
    #Concatate ICD9 and ICD10 results
    elix_df = pd.concat(
        [elix_9_df[['id','comorbidity_score','age_adj_comorbidity_score']],elix_10_df[['id','comorbidity_score','age_adj_comorbidity_score']]],
        ignore_index=True
    )

    #Rename columns for our convention
    elix_df.rename(columns={'id':'encounter_block','comorbidity_score':'elixhauser','age_adj_comorbidity_score':'elixhauser_age_adj'},inplace=True)

    block_df = pd.merge(
        block_df,
        elix_df,
        on='encounter_block',
        how='left'
    )
    #Fill values for empty encnounters.
    #Justified because ALL encounters have ICD codes, missing implies no comorbidities because we removed the primary admisison diagnosis.
    block_df = block_df.fillna(value={'elixhauser':0,'elixhauser_age_adj':0})

    del diag_df, diag_9_df, diag_10_df, elix_9_df, elix_10_df, elix_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}") 

Number of encnounters with comorbidities in both ICD codes: 0
Block Length: 32276
Unique Encounter Block: 32126


In [27]:
#Save it to a parquet file
block_df.to_parquet('block_compiled_no_hourly.parquet')

In [28]:
#Add Hourly Block by Block Data
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

#Load up the hourly block final data
hourly_df = pd.read_parquet(os.path.join(helper.path_mob,"intermediate","final_df_w_criteria.parquet"))
hourly_df['encounter_block'] = hourly_df['encounter_block'].astype(str)
hourly_df = hourly_df.sort_values(['encounter_block', 'time_from_vent'])#Ensure proper chronological order
block_final = pd.read_parquet('block_compiled_no_hourly.parquet')
block_final['encounter_block'] = block_final['encounter_block'].astype(str)

#Load and filter Hospitalizations data from CLIF
hosp_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_hospitalization.parquet"))
hosp_df = hosp_df[hosp_df['patient_id'].isin(block_final['patient_id'])].reset_index()
hosp_df['admission_dttm'] = helper.ensure_datetime_est(hosp_df['admission_dttm'])

#Aadd encounter_block to the hospitalization data frame.
hosp_df = hosp_df.sort_values('admission_dttm')
hosp_df = hosp_df.merge(
    block_final[['patient_id','encounter_block','block_vent_start_dttm']].drop_duplicates(),
    on='patient_id',
    how='left').reset_index()

#Load assessments
rass_df = pd.read_parquet(os.path.join(helper.path_clif,"clif_patient_assessments.parquet"))
rass_df = rass_df[rass_df['assessment_category'] == 'RASS']
rass_df.rename(columns={'numerical_value': 'RASS'}, inplace=True)

#Remove the ones not in cohort, add encounter_block and vent start time as reference
rass_df = rass_df.merge(hosp_df[['hospitalization_id','encounter_block','block_vent_start_dttm']],on='hospitalization_id', how='right').reset_index()

#fix timezone issues
rass_df['recorded_dttm'] = helper.ensure_datetime_est(rass_df['recorded_dttm'])

#Add date and time column
rass_df['recorded_date'] = rass_df['recorded_dttm'].dt.date
rass_df['recorded_hour'] = rass_df['recorded_dttm'].dt.hour
rass_df = rass_df[['encounter_block', 'recorded_date', 'recorded_hour', 'RASS']].copy().drop_duplicates()
hourly_df['recorded_date'] = pd.to_datetime(hourly_df['recorded_date'])
rass_df['recorded_date'] = pd.to_datetime(rass_df['recorded_date'])
hourly_df['recorded_hour'] = hourly_df['recorded_hour'].astype(int)

#Add RASS data to hourly blocks
hourly_df = pd.merge_ordered(
    hourly_df,
    rass_df,
    on=['encounter_block', 'recorded_date', 'recorded_hour'],
    how='left')
del rass_df

#Forward fill RASS
hourly_df['RASS'] = (
    hourly_df
    .groupby(['encounter_block'])['RASS']
    .transform(lambda x: x.ffill(limit=12)) #Note this is because it was already sorted above when the hourly DF was called.
)

#Add labels to divide into the chunks we want
hourly_df['time_from_vent'] = hourly_df['time_from_vent'].astype(int)
hourly_df['time_window'] = hourly_df['time_from_vent'].apply(helper.classify_time_window)

#Define coma
hourly_df['coma'] = hourly_df['RASS'] < -2
hourly_df['coma'] = hourly_df['coma'].astype(int)

#Group by encounter block and by time window, calculate total time in coma
grouped_coma_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['coma']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_coma_df.rename(columns={
    '0-24h': 'coma_0_24h',
    '24-48h': 'coma_24_48h',
    '48-72h': 'coma_48_72h'
}, inplace=True)
grouped_coma_df.drop(columns=['prior24h'], inplace=True, errors='ignore')

block_final = pd.merge(
    block_final,
    grouped_coma_df,
    on='encounter_block',
    how='left'
)

del grouped_coma_df

#Group by encounter block and by time window, calculate max pressor dose
grouped_pressor_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['ne_calc_max']
    .max()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_pressor_df.rename(columns={
    '0-24h': 'pressor_0_24h',
    '24-48h': 'pressor_24_48h',
    '48-72h': 'pressor_48_72h'
}, inplace=True)
grouped_pressor_df.drop(columns=['prior24h'], inplace=True, errors='ignore')

#Convert to boolean
#Note that the hourly block data does not extend to prior to IMV initiation. For that, we use the sofa_score.py output.
grouped_pressor_df['pressor_0_24h'] = grouped_pressor_df['pressor_0_24h'] > 0
grouped_pressor_df['pressor_24_48h'] = grouped_pressor_df['pressor_24_48h'] > 0
grouped_pressor_df['pressor_48_72h'] = grouped_pressor_df['pressor_48_72h'] > 0

block_final = pd.merge(
    block_final,
    grouped_pressor_df,
    on='encounter_block',
    how='left'
)

del grouped_pressor_df

#Group by encounter block and by time window, calculate sum of paralytics flags
grouped_para_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['paralytics_flag']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_para_df.rename(columns={
    '0-24h': 'paralytics_0_24h',
    '24-48h': 'paralytics_24_48h',
    '48-72h': 'paralytics_48_72h'
}, inplace=True)
grouped_para_df.drop(columns=['prior24h'], inplace=True, errors='ignore')

#Convert to boolean
grouped_para_df['paralytics_0_24h'] = grouped_para_df['paralytics_0_24h'] >= 4
grouped_para_df['paralytics_24_48h'] = grouped_para_df['paralytics_24_48h'] >= 4
grouped_para_df['paralytics_48_72h'] = grouped_para_df['paralytics_48_72h'] >= 4

block_final = pd.merge(
    block_final,
    grouped_para_df,
    on='encounter_block',
    how='left'
)

del grouped_para_df


## Group by encounter block and by time window, calculate total time in yellow eligibility all hours
#hourly_df['yellow_not_all_green'] = hourly_df['any_yellow_or_green_no_red_all_hours'].astype(int)
grouped_yellow_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['any_yellow_or_green_no_red_all_hours']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_yellow_df.rename(columns={
    '0-24h': 'yellow_0_24h',
    '24-48h': 'yellow_24_48h',
    '48-72h': 'yellow_48_72h'
}, inplace=True)
grouped_yellow_df.drop(columns=['prior24h'], inplace=True, errors='ignore')

block_final = pd.merge(
    block_final,
    grouped_yellow_df,
    on='encounter_block',
    how='left'
)

del grouped_yellow_df

###VENT FREE DAYS
#If dead, 0
#Otherwise uses the last hour of IMV on the hourly data_frame
vent_df = hourly_df[['encounter_block','time_from_vent','hourly_on_vent']]
vent_df['time_from_vent'] = vent_df['time_from_vent'].astype(int)
vent_df['hourly_on_vent'] = vent_df['hourly_on_vent'].astype(bool)
#Keep only values within 28-days and on-vent
vent_df = vent_df[(vent_df['time_from_vent'] <= 28*24) & vent_df['hourly_on_vent'] ]
#Get the MAX hour and merge it into DF
last_vent_df = (
    vent_df.groupby('encounter_block')['time_from_vent']
    .max()
    .reset_index()
)
last_vent_df['time_from_vent'] = last_vent_df['time_from_vent'].astype(int)
last_vent_df.rename(columns={'time_from_vent':'last_hour_on_vent'}, inplace=True)
block_final = pd.merge(
    block_final,
    last_vent_df,
    on='encounter_block',
    how='left'
)
del last_vent_df
del vent_df
#Get an 1 for patients alive at 28-days.
block_final['alive28'] = block_final['date_of_death'].isna() | ((block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() >= (28*24*60*60))
block_final['alive28'] = block_final['alive28'].astype(int)
#Calcute VFD
block_final['vent_free_days'] = block_final['alive28'] * (28 - block_final['last_hour_on_vent']/24)
block_final = block_final.drop(columns=['alive28','last_hour_on_vent'])

#REINTUBATIONS
vent_df = hourly_df[['encounter_block','time_from_vent','hourly_on_vent']] #Note it is already sorted in the proper order

#Intubation count for hospitalization only, not including other hospitalizations.
def count_intubations(series):
    """Counts the number of re-intubations as 0->1 transitions."""
    return ((series.shift(periods=1, fill_value=0) == 1) & (series == 0)).sum()

intubation_count_df = (
    vent_df
    .groupby('encounter_block')['hourly_on_vent']
    .apply(count_intubations)
    .reset_index()
    .rename(columns={'hourly_on_vent': 'intubation_count'})
)
block_final = pd.merge(
    block_final,
    intubation_count_df,
    on='encounter_block',
    how='left'
)
block_final['reintubation'] = (block_final['intubation_count'] > 1)

del hourly_df, intubation_count_df


print(f"Block Length: {len(block_final)}")
print(f"Unique Encounter Block: {block_final['encounter_block'].nunique()}")

#Save it to a parquet file
block_final.to_parquet('block_compiled_with_hourly.parquet')


Block Length: 32276
Unique Encounter Block: 32126


In [29]:
#FINALIZE DATA FRAME FOR STATISTICAL ANALYSIS
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np

#Upload final file
block_final = pd.read_parquet('block_compiled_with_hourly.parquet')

#Change relevant DTTM values to hours/days
block_final['imv_to_discharge_days'] = (block_final['discharge_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(24*3600)
block_final['imv_to_end_hours'] = (block_final['block_vent_end_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(3600)
block_final['adm_to_imv_hours'] = (block_final['block_vent_start_dttm'] - block_final['admission_dttm']).dt.total_seconds()/3600
block_final['imv_to_death_days'] = (block_final['death_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(24*3600)
block_final['adm_to_discharge_days'] = (block_final['discharge_dttm'] - block_final['block_first_vital_dttm']).dt.total_seconds()/(24*3600)
block_final['icu_to_imv_hours'] = (block_final['block_vent_start_dttm'] - block_final['icu_in_dttm']).dt.total_seconds()/(3600) #Positive if in ICU first before IMV.

#Add in a dichotomized outcomes variables
block_final['pt_ever'] = block_final['Time_first_PT'].notna()
block_final['pt_post48_IMV'] = block_final['Time_first_PT'].notna() & (block_final['Time_first_PT'] <= 48)
block_final['pt_post48_Yellow'] = block_final['delayed_yellow_PT'].notna() & (block_final['delayed_yellow_PT'] <= 48)
block_final['pt_pre24_IMV'] = block_final['Time_last_PT'].notna() & (block_final['Time_last_PT'] >= -24)
block_final['pt_pre24_or_post48_IMV'] = block_final['pt_pre24_IMV'] | block_final['pt_post48_IMV']
block_final['pt_pre24_or_post48_Yellow'] = block_final['pt_pre24_IMV'] | block_final['pt_post48_Yellow']
block_final['yellow_ever'] = block_final['yellow_time_eligibility'].notna()
block_final['yellow_post48_IMV'] = block_final['yellow_ever'] & (block_final['yellow_time_eligibility'] <= 48)
block_final['extubated_at_pt'] = block_final['imv_to_end_hours'] <= block_final['Time_first_PT']
block_final['is_dead'] = block_final['is_dead'].astype(bool)
block_final['pt_between_ICU_IMV'] = block_final['Time_first_PT'] < (-1*block_final['icu_to_imv_hours'])

# Add Hospital mortality: TRUE if Death_dttm < discharge_dttm or (discharge category is hospice or dead) 
block_final["is_dead_hosp"] = (
    (block_final["death_dttm"] <= block_final["discharge_dttm"]) |
    (block_final["discharge_category"].str.lower().isin(["Hospice", "Expired"]))
)
#30-day mortality
block_final['is_dead_30'] = (block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() <= (30*24*60*60)
#365-day mortality
block_final['is_dead_365'] = (block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() <= (365*24*60*60)

#remove duplicated encounter blocks (for stitched encounters) and save
block_final = block_final.drop_duplicates(subset=['encounter_block'], keep='first')
block_final = block_final.drop(columns=['hospitalization_id'])
block_final.to_parquet('block_with_outcomes.parquet')

In [30]:
with open("column_checkpoint.txt", "w") as text_file:
    for col_name in block_final.sort_index(axis=1).columns.tolist():
        text_file.write(col_name + '\n')

In [31]:
##CLUSTERING OF CATEGORICAL DATA & ORDERING COLUMNS

import pandas as pd
import numpy as np

df = pd.read_parquet('block_with_outcomes.parquet')

#Individualy and manually cluster all the categorical variables we want to cluster
print("LANGUAGE PRE:")
print(df['language'].value_counts(dropna=False))
keep = {"English","Spanish"}
set_mask = df['language'].notna() & (~df['language'].eq("None"))
df["language"] = np.where(df["language"].isin(keep), df["language"], "Other")
df["language"] = np.where(set_mask, df['language'], None)
print("LANGUAGE POST:")
print(df['language'].value_counts(dropna=False)) #Print results

print("RACE PRE:")
print(df['race_category'].value_counts(dropna=False))
keep = {"White", "Black or African American"}
set_mask = df['race_category'].notna() & (~df['race_category'].eq("Unknown"))
df["race_category"] = np.where(df["race_category"].isin(keep), df["race_category"], "Other")
df["race_category"] = np.where(set_mask, df['race_category'], None)
print("RACE POST:")
print(df['race_category'].value_counts(dropna=False)) #Print results

print("INSURANCE PRE:")
print(df['insurance'].value_counts(dropna=False))
keep = {"Medicare", "Medicaid","Private"}
set_mask = df['insurance'].notna() & (~df['insurance'].eq("Unknown"))
df["insurance"] = np.where(df["insurance"].isin(keep), df["insurance"], "None/Other")
df["insurance"] = np.where(set_mask, df['insurance'], None)
print("INSURANCE POST:")
print(df['insurance'].value_counts(dropna=False)) #Print results

#This just converts "Unknown" to None for better missingness tracking.
set_mask = df['ethnicity_category'].notna() & (~df['ethnicity_category'].eq("Unknown"))
df["ethnicity_category"] = np.where(set_mask, df['ethnicity_category'], None)

print("ICU TYPE PRE:")
print(df['ICU_type'].value_counts(dropna=False))
mapping = {
    "Medical Intensive Care Unit (MICU)": "Medical ICU",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)": "Medical ICU",
    "Intensive Care Unit (ICU)": "Medical ICU", 
    "Surgical Intensive Care Unit (SICU)": "Surgical ICU",
    "Coronary Care Unit (CCU)": "Cardiac ICU",
    "Cardiac Vascular Intensive Care Unit (CVICU)": "Cardiac ICU",
    "Trauma SICU (TSICU)": "Other",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)": "Other"
}
df['ICU_type'] = df['ICU_type'].map(mapping)
df['ICU_type'] = np.where(df['ICU_type'].notna(), df['ICU_type'], None)
print("ICU TYPE POST:")
print(df['ICU_type'].value_counts(dropna=False))

print("DISCHARGE PRE:")
print(df['discharge_category'].value_counts(dropna=False))
mapping = {
    "Home": "Home",
    "Against Medical Advice (AMA)": "Home",
    "Assisted Living": "Home",
    "Hospice": "Hospice", "Expired": "Expired",
    "Skilled Nursing Facility (SNF)": "Rehabilitation",
    "Acute Inpatient Rehab Facility": "Rehabilitation",
    "Psychiatric Hospital": "Other",
    "Acute Care Hospital": "Other",
    "Long Term Care Hospital (LTACH)": "Other",
    "Other": "Other", "Missing": "Missing"
}
df['discharge_category'] = df['discharge_category'].map(mapping)
print("DISCHARGE POST:")
print(df['discharge_category'].value_counts(dropna=False))


print("ADMISSION SOURCE PRE:")
print(df['admission_source'].value_counts(dropna=False))
##THIS IS A PLACEHOLDER, WE NEED TO UPDATE MIMIC to CLIF MAPPINGS TO GET ADMISSION_TYPE_CATEGORY instead.
mapping = {
    "EW EMER.": "Emergency/Urgent",
    "URGENT": "Emergency/Urgent",
    "EU OBSERVATION": "Observation",
    "OBSERVATION ADMIT": "Observation",
    "DIRECT OBSERVATION": "Observation",
    "SURGICAL SAME DAY ADMISSION": "Surgery",
    "DIRECT EMER.": "Direct",
    "ELECTIVE": "Direct",
    "AMBULATORY OBSERVATION": "Observation",
}
df['admission_source'] = df['admission_source'].map(mapping)
print("ADMISSION SOURCE POST:")
print(df['admission_source'].value_counts(dropna=False))

print("MINIC Race PRE:")
print(df['mimic_race'].value_counts(dropna=False))
mapping = {
    "WHITE":"White",
    "UNKNOWN":"Missing",
    "BLACK/AFRICAN AMERICAN":"Black/African American",
    "OTHER":"Other",
    "UNABLE TO OBTAIN":"Missing",
    "WHITE - OTHER EUROPEAN":"White",
    "HISPANIC/LATINO - PUERTO RICAN":"Hispanic/Latino",
    "ASIAN":"Other",
    "ASIAN - CHINESE":"Other",
    "HISPANIC/LATINO - DOMINICAN":"Hispanic/Latino",
    "WHITE - RUSSIAN":"White",
    "PATIENT DECLINED TO ANSWER":"Missing",
    "HISPANIC OR LATINO":"Hispanic/Latino",
    "BLACK/CAPE VERDEAN":"Black/African American",
    "BLACK/CARIBBEAN ISLAND":"Black/African American",
    "PORTUGUESE":"White",
    "BLACK/AFRICAN":"Black/African American",
    "ASIAN - SOUTH EAST ASIAN":"Other",
    "WHITE - EASTERN EUROPEAN":"White",
    "ASIAN - ASIAN INDIAN":"Other",
    "AMERICAN INDIAN/ALASKA NATIVE":"Other",
    "WHITE - BRAZILIAN":"White",
    "HISPANIC/LATINO - GUATEMALAN":"Hispanic/Latino",
    "HISPANIC/LATINO - SALVADORAN":"Hispanic/Latino",
    "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER":"Other",
    "HISPANIC/LATINO - CUBAN":"Hispanic/Latino",
    "ASIAN - KOREAN":"Other",
    "HISPANIC/LATINO - HONDURAN":"Hispanic/Latino",
    "HISPANIC/LATINO - MEXICAN":"Hispanic/Latino",
    "MULTIPLE RACE/ETHNICITY":"Other",
    "SOUTH AMERICAN":"Hispanic/Latino",
    "HISPANIC/LATINO - CENTRAL AMERICAN":"Hispanic/Latino",
    "HISPANIC/LATINO - COLUMBIAN":"Hispanic/Latino"
}
df['mimic_race'] = df['mimic_race'].map(mapping)
print("MIMIC Race POST:")
print(df['mimic_race'].value_counts(dropna=False))

df.to_parquet('block_with_outcomes_clustered.parquet')


LANGUAGE PRE:
language
English                   29049
Spanish                    1021
Chinese                     355
Russian                     304
Portuguese                  241
Haitian                     202
None                        191
Kabuverdianu                184
Vietnamese                  100
Other                        91
Italian                      69
Arabic                       55
Modern Greek (1453-)         55
Khmer                        33
Persian                      27
Korean                       27
Polish                       24
Hindi                        23
American Sign Language       19
Thai                         13
French                       12
Amharic                      12
Bengali                       9
Armenian                      8
Somali                        1
Japanese                      1
Name: count, dtype: int64
LANGUAGE POST:
language
English    29049
Other       1865
Spanish     1021
None         191
Name: count, dtype: int64
R

In [32]:
#RUN STATS FOR TABLE 1
import pandas as pd
import scipy.stats as stats
import numpy as np
import os

df = pd.read_parquet('block_with_outcomes_clustered.parquet')
column_order = pd.read_csv("column_def.csv")
my_cols = column_order['name'].tolist()
column_order = column_order.set_index('name')

#Exclusion criteria
df = df[~df['pt_pre24_IMV']]

#Convert outcome to a categorical column
early_col = "pt_post48_IMV"
df["early_PT"] = np.where(df[early_col], "early_PT", "no_early_PT")
n_total = df["encounter_block"].count()
n_early = df[early_col].sum()
n_not = n_total - n_early

file_path = "table1.csv"
if os.path.exists(file_path):
    os.remove(file_path)

with open(file_path, mode="w") as file:
    file.write(f",,Overall,Early PT, No Early PT, P-value, Missing")
    for col in my_cols:
        lab = column_order.loc[col,"description"]
        if col == 'encounter_block':
            file.write(f"\nN,,{n_total},{n_early},{n_not},")
        elif df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):
            file.write(f"\n{lab}")
            cats = df[col].unique()
            p_df = pd.pivot_table(df[["encounter_block",col,"early_PT"]], index=col, columns="early_PT", values="encounter_block", aggfunc='count')
            chi2_stat, p_value, dof, expected = stats.chi2_contingency(p_df.to_numpy())
            for cc in cats:
                if cc: #Because there is a NAN category
                    cc_all = round(100*p_df.loc[cc,].sum()/df[col].count(),1)
                    cc_early = round(100*p_df.loc[cc,'early_PT'].sum()/p_df['early_PT'].sum(),1)
                    cc_not = round(100*p_df.loc[cc,'no_early_PT'].sum()/p_df['no_early_PT'].sum(),1)
                    file.write(f"\n,{cc},{cc_all}%,{cc_early}%,{cc_not}%")
            file.write(f", {round(p_value,5)}")
        elif df[col].dtype == "bool" or df[col].dtype == "boolean":
            if df[col].sum(skipna=True) > 0:#Pivot table does not work if there are no true values
                sub_df = df[df[col].notna()]
                sub_df['flag'] = np.where(sub_df[col],"TRUE", "FALSE")
                p_df = pd.pivot_table(sub_df[["encounter_block","flag","early_PT"]], index="flag", columns="early_PT", values="encounter_block", aggfunc='count')
                chi2_stat, p_value, dof, expected = stats.chi2_contingency(p_df.to_numpy())
                cc_all = round(100*p_df.loc["TRUE",].sum()/sub_df[col].count(),1)
                cc_early = round(100*p_df.loc["TRUE",'early_PT'].sum()/p_df['early_PT'].sum(),1)
                cc_not = round(100*p_df.loc["TRUE",'no_early_PT'].sum()/p_df['no_early_PT'].sum(),1)
                file.write(f"\n{lab},,{cc_all}%,{cc_early}%,{cc_not}%,{round(p_value,5)}")
            else:
                file.write(f"\n{lab},,0.00%,0.00%,0.00%,N/A")
        elif pd.api.types.is_numeric_dtype(df[col]):
            cc_all = df[col].dropna()
            cc_early = df.loc[df[early_col], col]
            cc_early = cc_early.dropna()
            cc_not = df.loc[~df[early_col], col]
            cc_not = cc_not.dropna()
            rank_stat, p_value = stats.ranksums(cc_early.tolist(), cc_not.tolist())
            file.write(f"\n{lab} (Med & IQR),,{round(cc_all.median(),2)}  ({round(cc_all.quantile(0.25),2)} - {round(cc_all.quantile(0.75),2)})")
            file.write(f",{round(cc_early.median(),2)}  ({round(cc_early.quantile(0.25),2)} - {round(cc_early.quantile(0.75),2)})")
            file.write(f",{round(cc_not.median(),2)}  ({round(cc_not.quantile(0.25),2)} - {round(cc_not.quantile(0.75),2)})")
            file.write(f",{round(p_value,5)}")
        else:
            file.write(f"\n{lab},ERROR,,,,,")
        #Missing data column
        mis_pct = round(100*sum(df[col].isna())/n_total,2)
        file.write(f",{mis_pct}%")
            
df = df[my_cols]
df.to_parquet('block_for_stats.parquet')
df.to_csv('block_for_stats.csv')